# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, zero-shot LOKO validation, baselines, MOO, figures.

**Run cells in order.**

In [21]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

Cloning into '2601_chip_paper'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 115 (delta 49), reused 91 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 2.70 MiB | 7.80 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/2601_chip_paper/2601_chip_paper


In [22]:
# Cell 2: Pull latest code and install dependencies
!git pull
!uv sync

Already up to date.
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 80 packages in 1ms
Prepared 13 packages in 110ms                                            
Installed 75 packages in 1.02s0.post0                       
 + about-time==4.2.1
 + alive-progress==3.3.0
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + autograd==1.8.0
 + cffi==2.0.0
 + click==8.3.3
 + cma==4.4.4
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + deprecated==1.3.1
 + executing==2.2.1
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.2.0
 + graphemeu==0.7.2
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.5.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mdurl==0.1.2
 + moocore==0.3.1
 + mpmath==1.3.0
 + nest-asyncio=

In [24]:
!uv add torch

Resolved 98 packages in 205ms                                        
Installed 29 packages in 562ms                                   
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.29.7
 + nvidia-nvjitlink==13.0.88
 + nvidia-nvshmem-cu13==3.4.5
 + nvidia-nvtx==13.0.85
 + setuptools==81.0.0
 + sympy==1.14.0
 + torch==2.12.0
 + triton==3.7.0
 + typing-extensions==4.15.0


In [25]:
# Cell 3: Monotonicity ground truth audit (Paper Section 4.1)
!uv run python cli.py analysis monotonicity --max-groups 5000

Loading data/processed/master.parquet...
Rows with numeric param_3 and valid targets: 33746
Total groups: 11703

Monotonicity Audit Summary (4920 consecutive pairs)
  Both hold (area↑ AND latency↓):   2539 (51.6%)
  Area only (area↑, latency↗):         8 (0.2%)
  Latency only (area↓, latency↓):   2363 (48.0%)
  Neither:                             10 (0.2%)

Conclusion: joint monotonicity holds in 51.6% of pairs.
The HINN penalty acts as a SOFT REGULARIZER (Bayesian prior), not a hard constraint.
Include this analysis in Section 4.1 of the manuscript.

Per-group audit saved to results/data/monotonicity_audit.csv


In [ ]:
# Cell 4: Train HINN (Random Split mode)
!uv run python cli.py train train --epochs 500 --batch-size 1024 --seed 42

Seed: 42  |  Epochs: 500  |  Batch: 1024
Loading data...
Splits — Train: 35192, Val: 4400, Test: 4400
Scalers saved.
Device: cuda

Epoch  | LR       | Lambda | Train MSE  | Val MSE    | Val R2   | CVR (%) 
---------------------------------------------------------------------------
1      | 0.00100  | 0.00   | 0.6542     | 0.5994     | 0.4144   | 27.23   
10     | 0.00100  | 0.00   | 0.4730     | 0.4769     | 0.5317   | 35.76   
20     | 0.00100  | 0.00   | 0.3058     | 0.2863     | 0.7126   | 37.24   
30     | 0.00099  | 0.00   | 0.2466     | 0.2445     | 0.7526   | 36.49   
40     | 0.00098  | 0.04   | 0.2189     | 0.2276     | 0.7687   | 39.50   
50     | 0.00098  | 0.09   | 0.1926     | 0.2115     | 0.7846   | 38.55   
60     | 0.00097  | 0.13   | 0.1762     | 0.1987     | 0.7973   | 36.01   
70     | 0.00095  | 0.17   | 0.1588     | 0.1998     | 0.7964   | 33.70   
80     | 0.00094  | 0.21   | 0.1447     | 0.1860     | 0.8105   | 40.58   


## Leave-One-Kernel-Out (LOKO) Validation
This is the primary scientific novelty: testing zero-shot generalization on unseen hardware.

In [ ]:
# Cell 5: LOKO Sweep (example: hold out stencil2d)
!uv run python cli.py train train --epochs 500 --loko stencil2d --seed 42

In [ ]:
# Cell 6: Train baselines (XGBoost + Vanilla MLP)
!uv run python cli.py train baselines --epochs 500 --seed 42

In [ ]:
# Cell 7: Multi-Objective Optimization
!uv run python cli.py moo run

In [ ]:
# Cell 8: Generate Figures
!uv run python cli.py plot training-dynamics
!uv run python cli.py plot pareto
!uv run python cli.py plot monotonicity
!uv run python cli.py plot comparison

## Push Results to GitHub
Requires `GITHUB_PAT` to be set in Colab Secrets (left sidebar, key icon).

In [ ]:
# Cell 9: Authenticated Push
import os
from google.colab import userdata

try:
    pat = userdata.get('GITHUB_PAT')
    repo_url = f"https://{pat}@github.com/sattary/2601_chip_paper.git"

    !git config --global user.email "thesattary@gmail.com"
    !git config --global user.name "sat"
    !git add results/ models/  # Only add results and model weights
    !git commit -m "HINN training results from Colab sync"
    !git remote set-url origin {repo_url}
    !git push origin master
    print("Results successfully pushed to GitHub!")
except Exception as e:
    print(f"Sync failed: {e}")
    print("Check that GITHUB_PAT is added to Colab Secrets.")